In [7]:
import math
from collections import namedtuple
import ipywidgets as widgets
from IPython.display import display

# ==========================================
# 1. คลาส Game และ Alpha-Beta Search
# ==========================================
infinity = math.inf

class Game:
    def actions(self, state): raise NotImplementedError
    def result(self, state, move): raise NotImplementedError
    def utility(self, state, player): raise NotImplementedError
    def terminal_test(self, state): return not self.actions(state)
    def to_move(self, state): return state.to_move

def alphabeta_search(state, game):
    player = game.to_move(state)

    def max_value(state, alpha, beta):
        if game.terminal_test(state):
            return game.utility(state, player)
        v = -infinity
        for a in game.actions(state):
            v = max(v, min_value(game.result(state, a), alpha, beta))
            if v >= beta: return v
            alpha = max(alpha, v)
        return v

    def min_value(state, alpha, beta):
        if game.terminal_test(state):
            return game.utility(state, player)
        v = infinity
        for a in game.actions(state):
            v = min(v, max_value(game.result(state, a), alpha, beta))
            if v <= alpha: return v
            beta = min(beta, v)
        return v

    best_score = -infinity
    beta = infinity
    best_action = None
    for a in game.actions(state):
        v = min_value(game.result(state, a), best_score, beta)
        if v > best_score:
            best_score = v
            best_action = a
    return best_action

def alphabeta_player(game, state):
    return alphabeta_search(state, game)


# ==========================================
# 2. กฎของเกม Hexapawn
# ==========================================
GameState = namedtuple('GameState', 'to_move, utility, board, moves')

class Hexapawn(Game):
    def __init__(self, h=3, v=3):
        self.h = h
        self.v = v
        board = {}
        for c in range(1, v + 1):
            board[(1, c)] = 'W'
            board[(h, c)] = 'B'
            
        self.initial = GameState(
            to_move='W',
            utility=0,
            board=board,
            moves=self._compute_legal_moves(board, 'W')
        )

    def _compute_legal_moves(self, board, player):
        moves = []
        direction = 1 if player == 'W' else -1
        opponent = 'B' if player == 'W' else 'W'
        
        for (r, c), p in board.items():
            if p == player:
                next_r = r + direction
                if 1 <= next_r <= self.h and (next_r, c) not in board:
                    moves.append(((r, c), (next_r, c)))
                for next_c in [c - 1, c + 1]:
                    if 1 <= next_r <= self.h and 1 <= next_c <= self.v:
                        if board.get((next_r, next_c)) == opponent:
                            moves.append(((r, c), (next_r, next_c)))
        return moves

    def actions(self, state):
        return state.moves

    def result(self, state, move):
        if move not in state.moves: return state
        
        from_sq, to_sq = move
        board = state.board.copy()
        
        board[to_sq] = state.to_move
        if from_sq in board: del board[from_sq]
        
        next_player = 'B' if state.to_move == 'W' else 'W'
        next_moves = self._compute_legal_moves(board, next_player)
        utility = self._compute_utility(board, move, state.to_move, next_moves)
        
        return GameState(to_move=next_player, utility=utility, board=board, moves=next_moves)

    def _compute_utility(self, board, last_move, last_player, next_moves):
        _, (to_r, _) = last_move
        if last_player == 'W' and to_r == self.h: return 1
        if last_player == 'B' and to_r == 1: return -1
        
        next_player = 'B' if last_player == 'W' else 'W'
        if next_player not in board.values(): return 1 if last_player == 'W' else -1
        if len(next_moves) == 0: return 1 if last_player == 'W' else -1
        return 0

    def utility(self, state, player):
        return state.utility if player == 'W' else -state.utility

    def terminal_test(self, state):
        return state.utility != 0 or len(state.moves) == 0

    def to_move(self, state):
        return state.to_move


# ==========================================
# 3. หน้าต่างเกม Interactive UI
# ==========================================
class InteractiveHexapawnAI:
    def __init__(self):
        self.game = Hexapawn(3, 3)
        self.game_state = self.game.initial
        self.selected_pos = None
        self.buttons = {}
        self.setup_ui()
        
    def setup_ui(self):
        board_rows = []
        for r in range(1, 4):
            row_buttons = []
            for c in range(1, 4):
                btn = widgets.Button(
                    description='',
                    layout=widgets.Layout(width='80px', height='80px'),
                    style=widgets.ButtonStyle(font_size='22px', font_weight='bold')
                )
                btn.pos = (r, c)
                btn.on_click(self.on_cell_clicked)
                self.buttons[(r, c)] = btn
                row_buttons.append(btn)
            board_rows.append(widgets.HBox(row_buttons, layout=widgets.Layout(justify_content='center')))
            
        self.status_widget = widgets.HTML(value="<h3 style='color: blue; text-align: center;'>🎮 คิวของคุณ (W) - คลิกเลือกหมาก</h3>")
        
        reset_btn = widgets.Button(description='🔄 เริ่มเกมใหม่', button_style='success', layout=widgets.Layout(width='160px'))
        reset_btn.on_click(self.reset_game)
        
        self.container = widgets.VBox([
            widgets.HTML("<h2 style='text-align: center;'>♟️ Hexapawn vs AlphaBeta AI</h2>"),
            widgets.HTML("<p style='text-align: center;'>ผู้เล่น: <b>W (ขาว/แถวบน)</b> | AI: <b>B (ดำ/แถวล่าง)</b></p>"),
            self.status_widget,
            widgets.VBox(board_rows),
            widgets.HTML("<br>"),
            widgets.HBox([reset_btn], layout=widgets.Layout(justify_content='center'))
        ])
        self.update_display()

    def on_cell_clicked(self, button):
        if self.game.terminal_test(self.game_state) or self.game_state.to_move != 'W': return
            
        pos = button.pos
        if self.game_state.board.get(pos) == 'W':
            self.selected_pos = pos
            self.update_display()
            button.button_style = 'warning'
            self.status_widget.value = f"<h3 style='color: orange; text-align: center;'>เลือกหมากที่ {pos} แล้ว - คลิกเลือกช่องปลายทาง</h3>"
            return
            
        if self.selected_pos:
            move = (self.selected_pos, pos)
            if move in self.game_state.moves:
                self.game_state = self.game.result(self.game_state, move)
                self.selected_pos = None
                self.update_display()
                
                if self.game.terminal_test(self.game_state):
                    self.handle_game_over()
                    return
                
                self.status_widget.value = "<h3 style='color: gray; text-align: center;'>🤖 AI กำลังคำนวณตาเดิน...</h3>"
                ai_move = alphabeta_player(self.game, self.game_state)
                self.game_state = self.game.result(self.game_state, ai_move)
                self.update_display()
                
                if self.game.terminal_test(self.game_state):
                    self.handle_game_over()
                else:
                    self.status_widget.value = "<h3 style='color: blue; text-align: center;'>🎮 คิวของคุณ (W) - คลิกเลือกหมาก</h3>"
            else:
                self.selected_pos = None
                self.update_display()
                self.status_widget.value = "<h3 style='color: red; text-align: center;'>❌ เดินไม่ได้! กรุณาเลือกหมากใหม่</h3>"

    def update_display(self):
        for pos, btn in self.buttons.items():
            # แก้บั๊ก: ใช้เว้นวรรค ' ' แทนค่าว่างเปล่า '' เพื่อบังคับให้ล้างตัวอักษร
            piece = self.game_state.board.get(pos, ' ')
            btn.description = piece
            btn.button_style = 'info' if piece == 'W' else ('danger' if piece == 'B' else '')

    def handle_game_over(self):
        utility = self.game.utility(self.game_state, 'W')
        if utility == 1:
            self.status_widget.value = "<h3 style='color: green; text-align: center;'>🎉 คุณชนะ! ยินดีด้วย</h3>"
        else:
            self.status_widget.value = "<h3 style='color: red; text-align: center;'>🤖 AlphaBeta AI ชนะ!</h3>"

    def reset_game(self, button=None):
        self.game_state = self.game.initial
        self.selected_pos = None
        self.update_display()
        self.status_widget.value = "<h3 style='color: blue; text-align: center;'>🎮 คิวของคุณ (W) - คลิกเลือกหมาก</h3>"

    def display(self):
        return self.container

interactive_hexapawn = InteractiveHexapawnAI()
display(interactive_hexapawn.display())